# 02 – Zweite Datenquelle: BFS-Gemeindedaten und Join

**Lenny Frei** · Data Analytics, ZHAW, HS2026 · Stand 1. November 2026

Dieses Notebook erfüllt **Pflichtanforderung 2**: zwei Datenquellen zusammenführen
und die Integration dokumentieren.

**Was hier passiert:**

1. Die Gemeindedaten des Bundesamts für Statistik (BFS) einlesen
2. Sie vom Lang- ins Breitformat drehen
3. Die Analysemenge aus Metehans Strompreisdaten bilden
4. Beide über die BFS-Gemeindenummer verbinden
5. Prüfen, wer keinen Partner findet, wo Werte fehlen, und ob das zufällig ist
6. Die verbundene Tabelle für Metehan und Subraj speichern

**Primärquelle:** ElCom-Strompreise über den SPARQL-Endpunkt von LINDAS (Metehan)
**Zweitquelle:** Regionalporträts 2021, Kennzahlen aller Gemeinden (BFS)

### Bibliotheken laden

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

### Die BFS-Datei einlesen

Die Datei ist mit Semikolon getrennt und in UTF-8 **mit BOM** gespeichert. Das BOM ist
ein unsichtbares Zeichen am Dateianfang; ohne `encoding="utf-8-sig"` würde die erste
Spalte `﻿PERIOD_REF` statt `PERIOD_REF` heissen und kein Zugriff darauf funktionieren.
Alles wird zuerst als Text gelesen, damit pandas nichts vorschnell umwandelt.

In [2]:
bfs_roh = pd.read_csv(
    "daten/bfs/regionalportraets_2021.csv",
    sep=";", encoding="utf-8-sig", dtype=str,
)

print(f"Zeilen : {len(bfs_roh)}")
print(f"Spalten: {list(bfs_roh.columns)}")
bfs_roh.head()

Zeilen : 89093
Spalten: ['PERIOD_REF', 'PERIOD_COMP', 'CODE_REGION', 'REGION', 'INDICATORS', 'UNIT_MES', 'VALUE', 'STATUS']


,PERIOD_REF,PERIOD_COMP,CODE_REGION,REGION,INDICATORS,UNIT_MES,VALUE,STATUS
0,2019,NaN,CH,Schweiz,Ind_01_01,UM_6,8606033,A
1,2019,NaN,1,Aeugst am Albis,Ind_01_01,UM_6,1981,A
2,2019,NaN,2,Affoltern am Albis,Ind_01_01,UM_6,12303,A
3,2019,NaN,3,Bonstetten,Ind_01_01,UM_6,5572,A
4,2019,NaN,4,Hausen am Albis,Ind_01_01,UM_6,3751,A


### Das Problem: die Datei ist im Langformat

Erwartet hätte man eine Zeile pro Gemeinde. Stattdessen gibt es **eine Zeile pro
Gemeinde und Kennzahl**. Jede Gemeinde belegt also rund 41 Zeilen.

Hier alle Zeilen der Gemeinde Aeugst am Albis (BFS-Nummer 1):

In [3]:
beispiel = bfs_roh[bfs_roh.CODE_REGION == "1"]
print(f"Aeugst am Albis belegt {len(beispiel)} Zeilen.")
beispiel[["CODE_REGION", "REGION", "INDICATORS", "PERIOD_REF", "VALUE", "STATUS"]].head(8)

Aeugst am Albis belegt 41 Zeilen.


,CODE_REGION,REGION,INDICATORS,PERIOD_REF,VALUE,STATUS
1,1,Aeugst am Albis,Ind_01_01,2019,1981,A
2174,1,Aeugst am Albis,Ind_01_02,2010/2019,8.60745614,A
4347,1,Aeugst am Albis,Ind_01_03,2019,250.4424779,A
6520,1,Aeugst am Albis,Ind_01_08,2019,14.18475517,A
8693,1,Aeugst am Albis,Ind_01_04,2019,19.78798587,A
10866,1,Aeugst am Albis,Ind_01_05,2019,62.19081272,A
13039,1,Aeugst am Albis,Ind_01_06,2019,18.02120141,A
15212,1,Aeugst am Albis,Ind_01_09,2019,7.065354529,A


### Was bedeutet die Spalte STATUS?

Leere Werte sind hier nicht einfach Lücken, sondern begründet:

| Code | Bedeutung |
|---|---|
| `A` | Normaler Wert |
| `M` | Entfällt, weil trivial oder Begriffe nicht anwendbar |
| `Q` | **Entfällt aus Datenschutzgründen** |

`Q` ist wichtig: Das BFS unterdrückt Werte, wenn sie Rückschlüsse auf einzelne
Personen zuliessen. Das trifft vor allem **kleine Gemeinden**. Fehlende Werte sind
hier also nicht zufällig verteilt. Darauf kommen wir weiter unten zurück.

In [4]:
print(bfs_roh.STATUS.value_counts(dropna=False).to_string())
print(f"\nLeere Werte in VALUE: {bfs_roh.VALUE.isna().sum()}")

STATUS
A    84945
M     2842
Q     1306

Leere Werte in VALUE: 4148


### Zwei Kennzahlen aussortieren

Die Kennzahlen stammen aus **unterschiedlichen Bezugsjahren**, obwohl die Datei
«Regionalporträts 2021» heisst. Zwei davon beziehen sich auf den Zeitraum
**2004/2009** – Siedlungsflächen und deren Veränderung. Diese gegen Strompreise von
2027 zu stellen, wäre nicht zu verteidigen, darum fliegen sie raus.

Die Gesamtfläche von 2016 behalten wir: Gemeindeflächen ändern sich kaum, ausser bei
Fusionen, und Fusionsgemeinden sind ohnehin ausgeschlossen.

In [5]:
print(bfs_roh.groupby("INDICATORS").PERIOD_REF.first().value_counts().to_string())

AUSGESCHLOSSEN = ["Ind_04_02", "Ind_04_03"]   # Siedlungsflächen, Bezug 2004/2009
print(f"\nAusgeschlossen: {AUSGESCHLOSSEN}")

PERIOD_REF
2019         23
2018          9
2004/2009     4
1979/1985     2
2010/2019     1
2016          1
2020          1

Ausgeschlossen: ['Ind_04_02', 'Ind_04_03']


### Vom Langformat ins Breitformat: der Pivot

`pivot` dreht die Tabelle. Aus vielen Zeilen pro Gemeinde wird **eine** Zeile, und
jede Kennzahl bekommt eine **eigene Spalte**.

```
vorher (lang)                     nachher (breit)
gemeinde | kennzahl  | wert       gemeinde | einwohner | dichte
Aeugst   | einwohner | 1981       Aeugst   |      1981 |    251
Aeugst   | dichte    |  251       Affoltern|     12303 |   1281
Affoltern| einwohner |12303
Affoltern| dichte    | 1281
```

Das ist nötig, weil ein Join eine Zeile je Gemeinde braucht. Mit dem Langformat
würde jede Gemeinde beim Verbinden 41-mal auftauchen.

Drei Dinge passieren vorher:
- Die Zeile für die ganze Schweiz (`CODE_REGION == "CH"`) fällt weg, sie ist keine Gemeinde
- Die Gemeindenummer wird **Ganzzahl**, damit sie zu Metehans Daten passt
- `VALUE` wird in eine Zahl umgewandelt; was sich nicht umwandeln lässt, wird `NaN`

In [6]:
bfs = bfs_roh[bfs_roh.CODE_REGION != "CH"].copy()
bfs = bfs[~bfs.INDICATORS.isin(AUSGESCHLOSSEN)]

bfs["gemeinde_nr"] = bfs.CODE_REGION.astype("int64")
bfs["wert"] = pd.to_numeric(bfs.VALUE.str.replace(",", "."), errors="coerce")

# Kontrolle: jede Kombination aus Gemeinde und Kennzahl darf nur einmal vorkommen,
# sonst wuerde pivot scheitern.
doppelt = bfs.duplicated(["gemeinde_nr", "INDICATORS"]).sum()
print(f"Doppelte Gemeinde/Kennzahl-Paare: {doppelt}")

bfs_breit = bfs.pivot(index="gemeinde_nr", columns="INDICATORS", values="wert")
bfs_breit.insert(0, "gemeinde_bfs", bfs.groupby("gemeinde_nr").REGION.first())

print(f"Breitformat: {bfs_breit.shape[0]} Gemeinden x {bfs_breit.shape[1]-1} Kennzahlen")
bfs_breit.iloc[:5, :6]

Doppelte Gemeinde/Kennzahl-Paare: 0
Breitformat: 2172 Gemeinden x 39 Kennzahlen


INDICATORS,gemeinde_bfs,Ind_01_01,Ind_01_02,Ind_01_03,Ind_01_04,Ind_01_05
gemeinde_nr,,,,,,
1,Aeugst am Albis,1981.0,8.607456,250.442478,19.787986,62.190813
2,Affoltern am Albis,12303.0,10.927779,1161.756374,20.165813,61.667886
3,Bonstetten,5572.0,8.005427,749.932705,23.636037,59.834889
4,Hausen am Albis,3751.0,11.636905,275.808824,22.180752,59.824047
5,Hedingen,3778.0,10.791789,578.560490,21.625199,61.646374


### Das Format der Gemeindenummern prüfen

Der Join steht und fällt mit dem **Matching-Key**. Hier ist es die BFS-Gemeindenummer.
Gefährlich wäre, wenn eine Seite führende Nullen hätte: `04811` als Text würde `4811`
nie finden. Darum prüfen wir das ausdrücklich.

In [7]:
codes = bfs_roh[bfs_roh.CODE_REGION != "CH"].CODE_REGION
print(f"Stellenzahl der BFS-Codes: {sorted(int(x) for x in codes.str.len().unique())}")
print(f"Codes mit führender Null : {codes.str.startswith('0').sum()}")
print(f"Beide Seiten werden als Ganzzahl gelesen -> Formate passen zusammen.")

Stellenzahl der BFS-Codes: [1, 2, 3, 4]
Codes mit führender Null : 0
Beide Seiten werden als Ganzzahl gelesen -> Formate passen zusammen.


### Die Analysemenge aus den Strompreisdaten bilden

Gemäss den festgelegten Entscheiden der Gruppe:

- nur **Kategorie H4** (Durchschnittshaushalt, 4500 kWh im Jahr)
- nur **produkt = standard**, nicht `cheapest`
- bei mehreren Netzbetreibern pro Gemeinde der **Mittelwert**
- nur Gemeinden, die in **allen drei Jahren** vorkommen

Zusätzlich wird je Jahr eine Spalte `vergleichbar` gebildet: die Summe aus energie,
netznutzung, abgaben und netzzuschlag – **ohne** Messtarif. Grund: Den Messtarif gibt
es erst ab 2026. Wer `total` über die Zeit vergleicht, überschätzt den Anstieg 2027
um rund 1.67 Rappen.

In [8]:
TEILE = ["energie", "netznutzung", "abgaben", "netzzuschlag"]
DATEIEN = {2021: "Daten_Strompreise_2021.csv",
           2024: "Daten_Strompreise_2024.csv",
           2027: "Daten_Strompreise_2027.csv"}

pro_jahr = {}
for jahr, datei in DATEIEN.items():
    df = pd.read_csv(datei)
    df = df[(df.kategorie == "H4") & (df.produkt == "standard")].copy()
    df["vergleichbar"] = df[TEILE].sum(axis=1)
    pro_jahr[jahr] = df
    print(f"{jahr}: {len(df)} Zeilen, {df.gemeinde_nr.nunique()} Gemeinden")

kern = sorted(set(pro_jahr[2021].gemeinde_nr) &
              set(pro_jahr[2024].gemeinde_nr) &
              set(pro_jahr[2027].gemeinde_nr))
print(f"\nIn allen drei Jahren vorhanden: {len(kern)} Gemeinden")

2021: 2374 Zeilen, 2201 Gemeinden
2024: 2300 Zeilen, 2140 Gemeinden
2027: 2242 Zeilen, 2109 Gemeinden

In allen drei Jahren vorhanden: 2087 Gemeinden


Jetzt die Preise je Gemeinde zusammenfassen. Jede Gemeinde bekommt pro Jahr einen
Gesamtpreis und einen vergleichbaren Preis. Zusätzlich wird der Netzbetreiber
festgehalten – Subraj braucht ihn, weil die Preise laut Abschnitt 5b der
Projektanleitung pro Netzbetreiber gesetzt werden. Bei mehreren Betreibern pro
Gemeinde wird der mit der **kleinsten `netzbetreiber_id`** genommen; diese Regel ist
willkürlich, aber reproduzierbar, im Gegensatz zur Zeilenreihenfolge der Datei.

In [9]:
preise = pd.DataFrame(index=pd.Index(kern, name="gemeinde_nr"))

for jahr, df in pro_jahr.items():
    g = df.groupby("gemeinde_nr")[["total", "vergleichbar"]].mean()
    preise[f"total_{jahr}"] = g.total
    preise[f"vergleichbar_{jahr}"] = g.vergleichbar

basis = pro_jahr[2027].sort_values("netzbetreiber_id").groupby("gemeinde_nr").first()
preise.insert(0, "gemeinde", basis.gemeinde)
preise["netzbetreiber_id"] = basis.netzbetreiber_id
preise["netzbetreiber"] = basis.netzbetreiber

print(f"{len(preise)} Gemeinden, {preise.netzbetreiber_id.nunique()} Netzbetreiber")
preise.head()

2087 Gemeinden, 485 Netzbetreiber


,gemeinde,total_2021,vergleichbar_2021,total_2024,vergleichbar_2024,total_2027,vergleichbar_2027,netzbetreiber_id,netzbetreiber
gemeinde_nr,,,,,,,,,
1,Aeugst am Albis,17.122,17.121,31.609,31.608,23.872,22.219,486,Elektrizitätswerke des Kantons Zürich (EKZ)
2,Affoltern am Albis,17.122,17.121,31.609,31.608,23.872,22.219,486,Elektrizitätswerke des Kantons Zürich (EKZ)
3,Bonstetten,17.122,17.121,31.609,31.608,23.872,22.219,486,Elektrizitätswerke des Kantons Zürich (EKZ)
4,Hausen am Albis,17.122,17.121,31.609,31.608,23.872,22.219,486,Elektrizitätswerke des Kantons Zürich (EKZ)
5,Hedingen,17.122,17.121,31.609,31.608,23.872,22.219,486,Elektrizitätswerke des Kantons Zürich (EKZ)


### Der Join

`how="left"` bedeutet: Alle 2087 Gemeinden aus den Strompreisdaten bleiben erhalten.
Findet eine keinen Partner in den BFS-Daten, stehen dort leere Felder – so sieht man
die Verluste, statt dass sie stillschweigend verschwinden.

In [10]:
verbunden = preise.join(bfs_breit, how="left")

treffer = verbunden.gemeinde_bfs.notna()
print("=== Ergebnis des Joins ===")
print(f"Gemeinden aus den Strompreisdaten : {len(verbunden)}")
print(f"Partner in den BFS-Daten gefunden : {treffer.sum()} ({treffer.mean()*100:.2f} %)")
print(f"Ohne Partner                      : {(~treffer).sum()}")

if (~treffer).any():
    display(verbunden.loc[~treffer, ["gemeinde"]])
else:
    print("\nKeine einzige Gemeinde geht verloren.")

=== Ergebnis des Joins ===
Gemeinden aus den Strompreisdaten : 2087
Partner in den BFS-Daten gefunden : 2087 (100.00 %)
Ohne Partner                      : 0

Keine einzige Gemeinde geht verloren.


**Warum die Trefferquote 100 Prozent beträgt:** Unsere 2087 Gemeinden kommen alle
schon in den ElCom-Daten von 2021 vor – sonst wären sie nicht in der Schnittmenge
aller drei Jahre. Die BFS-Daten haben den Stand 1.1.2021. Alles, was 2021 existierte,
ist dort enthalten.

Die Beschränkung auf die 2087 Gemeinden löst damit **zwei** Probleme auf einmal: die
Vergleichbarkeit über die Zeit und den Altersunterschied zwischen den beiden Quellen.

### Die fünf Merkmale für das Modell

Nicht alle 39 Kennzahlen kommen ins Modell. Mit rund 500 effektiven Beobachtungen
würde eine Regression mit 39 Variablen Scheinbefunde liefern. Ausgewählt sind fünf,
die inhaltlich mit Netzkosten zu tun haben:

| Code | Name | Begründung |
|---|---|---|
| `Ind_01_01` | Einwohner | Grösse, Skaleneffekte beim Betreiber |
| `Ind_01_03` | Bevölkerungsdichte | je dünner besiedelt, desto mehr Leitung pro Anschluss |
| `Ind_04_01` | Fläche Total | Ausdehnung des Versorgungsgebiets |
| `Ind_06_03` | Beschäftigte Total | Anteil gewerblicher Last |
| `Ind_01_06` | Anteil 65+ | demografische Kontrollvariable |

Die übrigen Kennzahlen bleiben in der Tabelle und stehen für die EDA zur Verfügung.

In [11]:
MERKMALE = {"Ind_01_01": "einwohner",
            "Ind_01_03": "dichte",
            "Ind_04_01": "flaeche_total",
            "Ind_06_03": "beschaeftigte",
            "Ind_01_06": "anteil_65plus"}

uebersicht = pd.DataFrame({
    "fehlt_bei": [verbunden[c].isna().sum() for c in MERKMALE],
}, index=[f"{v} ({k})" for k, v in MERKMALE.items()])
uebersicht["Anteil"] = (uebersicht.fehlt_bei / len(verbunden) * 100).round(1)
uebersicht

,fehlt_bei,Anteil
einwohner (Ind_01_01),0,0.0
dichte (Ind_01_03),0,0.0
flaeche_total (Ind_04_01),0,0.0
beschaeftigte (Ind_06_03),88,4.2
anteil_65plus (Ind_01_06),0,0.0


### Fehlende Werte: trifft es zufällig?

Vier der fünf Merkmale sind vollständig. Nur bei den Beschäftigten fehlen Werte –
wegen der `Q`-Unterdrückung aus Datenschutzgründen. Die Frage ist, **wen** es trifft.

In [12]:
fehlt = verbunden.Ind_06_03.isna()
print("Beschäftigte Total – fehlende gegen vorhandene Werte:\n")
print(f"{'':<22}{'fehlend':>12}{'vorhanden':>12}")
print(f"{'Anzahl Gemeinden':<22}{fehlt.sum():>12}{(~fehlt).sum():>12}")
print(f"{'Median Einwohner':<22}{verbunden.loc[fehlt, 'Ind_01_01'].median():>12.0f}"
      f"{verbunden.loc[~fehlt, 'Ind_01_01'].median():>12.0f}")
print(f"{'Median Preis 2021':<22}{verbunden.loc[fehlt, 'total_2021'].median():>12.2f}"
      f"{verbunden.loc[~fehlt, 'total_2021'].median():>12.2f}")

Beschäftigte Total – fehlende gegen vorhandene Werte:

                           fehlend   vorhanden
Anzahl Gemeinden                88        1999
Median Einwohner               312        1717
Median Preis 2021            21.03       20.53


### Die zweite Verzerrung: die ausgeschlossenen Fusionsgemeinden

Es gibt eine zweite Gruppe, die aus der Analyse fällt: Gemeinden, die 2021 noch
existierten, bis 2027 aber fusioniert wurden. Beide Gruppen zusammen betrachtet zeigen,
ob unsere Auswahl systematisch verzerrt ist.

In [13]:
nur_2021 = sorted(set(pro_jahr[2021].gemeinde_nr) - set(kern))
fusion = (pro_jahr[2021][pro_jahr[2021].gemeinde_nr.isin(nur_2021)]
          .groupby("gemeinde_nr").agg(total=("total", "mean"))
          .join(bfs_breit))

behalten = verbunden[~fehlt]
zeilen = [
    ("Behalten (alle 5 Merkmale)", len(behalten),
     behalten.Ind_01_01.median(), behalten.total_2021.median()),
    ("Raus: Beschäftigte unterdrückt", fehlt.sum(),
     verbunden.loc[fehlt, "Ind_01_01"].median(), verbunden.loc[fehlt, "total_2021"].median()),
    ("Raus: Gemeindefusion", len(fusion),
     fusion.Ind_01_01.median(), fusion.total.median()),
]
verzerrung = pd.DataFrame(zeilen, columns=["Gruppe", "Anzahl",
                                           "Median Einwohner", "Median Preis 2021"])
verzerrung

,Gruppe,Anzahl,Median Einwohner,Median Preis 2021
0,Behalten (alle 5 Merkmale),1999,1717.0,20.533
1,Raus: Beschäftigte unterdrückt,88,312.5,21.032
2,Raus: Gemeindefusion,114,668.5,21.143


**Befund:** Beide ausgeschlossenen Gruppen sind deutlich **kleiner** und etwas
**teurer** als die behaltenen Gemeinden. Die beiden Verzerrungen ziehen in dieselbe
Richtung und verstärken sich. Zusammen fallen rund 17 Prozent der 2021 vorhandenen
Gemeinden heraus, überdurchschnittlich die kleinen und ländlichen.

Das ist für eine Aussage über Stadt und Land erheblich und gehört in die
Interpretation. Die ausführliche Fassung steht in `dokumentation_integration_lenny.md`.

### Die verbundene Tabelle speichern

Metehan und Subraj sollen direkt damit arbeiten können, ohne diesen Weg zu wiederholen.
Die Kennzahlen bekommen sprechende Namen statt der Codes `Ind_01_01` und so weiter.

In [14]:
AUSGABE = ["gemeinde", "netzbetreiber_id", "netzbetreiber",
           "total_2021", "total_2024", "total_2027",
           "vergleichbar_2021", "vergleichbar_2024", "vergleichbar_2027"]

tabelle = verbunden[AUSGABE + list(MERKMALE)].rename(columns=MERKMALE)

ziel = Path("daten/verbunden")
ziel.mkdir(parents=True, exist_ok=True)
tabelle.to_csv(ziel / "strompreise_gemeinden.csv", encoding="utf-8")

print(f"Gespeichert: {ziel / 'strompreise_gemeinden.csv'}")
print(f"{len(tabelle)} Zeilen, {len(tabelle.columns)} Spalten\n")
print(list(tabelle.columns))
tabelle.head()

Gespeichert: daten/verbunden/strompreise_gemeinden.csv
2087 Zeilen, 14 Spalten

['gemeinde', 'netzbetreiber_id', 'netzbetreiber', 'total_2021', 'total_2024', 'total_2027', 'vergleichbar_2021', 'vergleichbar_2024', 'vergleichbar_2027', 'einwohner', 'dichte', 'flaeche_total', 'beschaeftigte', 'anteil_65plus']


,gemeinde,netzbetreiber_id,netzbetreiber,total_2021,total_2024,total_2027,vergleichbar_2021,vergleichbar_2024,vergleichbar_2027,einwohner,dichte,flaeche_total,beschaeftigte,anteil_65plus
gemeinde_nr,,,,,,,,,,,,,,
1,Aeugst am Albis,486,Elektrizitätswerke des Kantons Zürich (EKZ),17.122,31.609,23.872,17.121,31.608,22.219,1981.0,250.442478,7.91,442.0,18.021201
2,Affoltern am Albis,486,Elektrizitätswerke des Kantons Zürich (EKZ),17.122,31.609,23.872,17.121,31.608,22.219,12303.0,1161.756374,10.59,6920.0,18.166301
3,Bonstetten,486,Elektrizitätswerke des Kantons Zürich (EKZ),17.122,31.609,23.872,17.121,31.608,22.219,5572.0,749.932705,7.43,1014.0,16.529074
4,Hausen am Albis,486,Elektrizitätswerke des Kantons Zürich (EKZ),17.122,31.609,23.872,17.121,31.608,22.219,3751.0,275.808824,13.60,1021.0,17.995201
5,Hedingen,486,Elektrizitätswerke des Kantons Zürich (EKZ),17.122,31.609,23.872,17.121,31.608,22.219,3778.0,578.560490,6.53,1478.0,16.728428


---

## Zusammenfassung

| | |
|---|---|
| Quelle 1 | ElCom-Strompreise über LINDAS (Metehan) |
| Quelle 2 | Regionalporträts 2021, BFS |
| Matching-Key | BFS-Gemeindenummer, beidseitig als Ganzzahl |
| Gemeinden in der Analyse | 2087 |
| Trefferquote des Joins | **100 Prozent** |
| Verlorene Datensätze | 0 |
| Fehlende Werte bei den fünf Merkmalen | nur Beschäftigte, 88 Gemeinden |
| Ausgeschlossen vor dem Join | 114 Fusionsgemeinden |

**Nächste Schritte:** Gemeindegrenzen für die Karte (Bonus 3), Kanton und
Sprachregion aus dem amtlichen Gemeindeverzeichnis.